In [1]:
# Imports
from matplotlib.ticker import FixedLocator, FuncFormatter
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [2]:
# Load data
df = pd.read_csv('./data/FinalTransactionReport.csv')

In [3]:
section_map = {
    # Seating Bowl
    '1': 'Seating Area',
    '2': 'Seating Area',
    '3': 'Seating Area',
    '4': 'Seating Area',
    '5': 'Seating Area',
    '6': 'Seating Area',
    '7': 'Seating Area',
    '8': 'Seating Area',
    '9': 'Seating Area',
    '10': 'Seating Area',
    '11': 'Seating Area',
    '12': 'Seating Area',
    '13': 'Seating Area',
    '14': 'Seating Area',
    '15': 'Seating Area',
    '16': 'Seating Area',
    '17': 'Seating Area',
    '18': 'Seating Area',
    '19': 'Seating Area',
    '20': 'Seating Area',
    '21': 'Seating Area',
    '22': 'Seating Area',
    '23': 'Seating Area',
    '24': 'Seating Area',

    # Club
    'CLUB1': 'Club',
    'CLUB2': 'Club',
    'CLUB3': 'Club',
    'CLUB4': 'Club',

    # Club SRO
    'CLUBSRO': 'Club SRO',

    # General SRO
    'STANDROOM': 'SRO',
    '0': 'SRO',
}

df['Section_Group'] = df['Section'].map(section_map).fillna('Other')


In [4]:
# Ensure timestamps are datetime
df['Event Timestamp'] = pd.to_datetime(df['Event Timestamp'])

section_level = (
    df.groupby(['Event ID', 'Section_Group'])
    .apply(lambda x: pd.Series({
        "event_id": x['Event ID'].iloc[0],
        "section_group": x['Section_Group'].iloc[0],
        "event_name": x['Event Name'].iloc[0],
        "month": x['Event Timestamp'].iloc[0].strftime("%B"),
        "day_of_week": x['Day of Week'].iloc[0],
        "is_weekend": int(x['Day of Week'].iloc[0] in ["Friday","Saturday","Sunday"]),

        "tickets_sold_section": x['Tickets Sold'].sum(),
        "revenue_section": x['Sales Total'].sum(),

        "avg_days_before_purchase": x['Days Before Event'].mean(),

        "temperature_f": x['temperature_f'].iloc[0],
        "humidity": x['humidity'].iloc[0],
        "weather_category": x['weather_category'].iloc[0],
    }))
    .reset_index(drop=True)
)


avg_section_tickets = section_level["tickets_sold_section"].mean()

section_level["section_demand_strength"] = (
    (section_level["tickets_sold_section"] - avg_section_tickets) / avg_section_tickets
)

/tmp/ipykernel_5980/1760208474.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


In [5]:
from interpret.glassbox import ExplainableBoostingRegressor
from sklearn.model_selection import train_test_split

feature_cols = [
    "event_name",
    "section_group",
    "month",
    "is_weekend",
    "avg_days_before_purchase",
    "temperature_f",
    "humidity",
    "weather_category",
]

section_models = {}

for group in section_level['section_group'].unique():
    subset = section_level[section_level['section_group'] == group]

    X = subset[feature_cols]
    y = subset['section_demand_strength']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = ExplainableBoostingRegressor(
        interactions=10,
        max_bins=256,
        outer_bags=1,
        inner_bags=0,
        learning_rate=0.01,
        max_rounds=5000,
        n_jobs=1,
        random_state=42
    )

    model.fit(X_train, y_train)

    section_models[group] = model

    print(f"Section Group: {group}")
    print("Train R^2:", model.score(X_train, y_train))
    print("Test  R^2:", model.score(X_test, y_test))
    print("------")



Section Group: Club
Train R^2: 0.047652137247496396
Test  R^2: -0.034389099930252076
------
Section Group: Club SRO
Train R^2: 0.45004424117243547
Test  R^2: -0.10516172840742088
------
Section Group: SRO
Train R^2: 0.3218723707030877
Test  R^2: -0.27299472684143344
------
Section Group: Seating Area
Train R^2: 0.5777990701613829
Test  R^2: 0.4470545984410045
------


In [6]:
from interpret import show

global_expl = section_models['SRO'].explain_global()
show(global_expl)

<!-- http://127.0.0.1:7952/138746861658048/ -->